In [0]:
import pandas as pd
import numpy as np

In [0]:
url  = '/Volumes/workspace/default/bronze/'
customer_path = url + 'olist_customers_dataset.csv'
order_path = url + 'olist_orders_dataset.csv'
seller_path = url + 'olist_sellers_dataset.csv'
product_path = url + 'olist_products_dataset.csv'
order_item_path = url + 'olist_order_items_dataset.csv'
order_review_path = url + 'olist_order_reviews_dataset.csv'
payment_path = url + 'olist_order_payments_dataset.csv'
geo_path = url + 'olist_geolocation_dataset.csv'

paths = {
    "customer_df": customer_path,
    "order_df": order_path,
    "seller_df": seller_path,
    "product_df": product_path,
    "order_item_df": order_item_path,
    "order_review_df": order_review_path,
    "payment_df": payment_path,
    "geo_df": geo_path,
}


dfs = {}
for name, path in paths.items():
    print(name, path)
    dfs[name] = spark.read.format("csv").option("header", True).option("inferSchema",True).load(path)
print(dfs)

Reading data from MongoDB

In [0]:
!pip install pymongo

In [0]:
import json
from pymongo import MongoClient
with open("/Volumes/workspace/default/bronze/mongodb.json",'r') as file:
    data = json.load(file)
    print(data)


In [0]:

uri = "mongodb://" + data[0]["username"] + ":" + data[0]["password"] + "@" + data[0]["hostname"] + ":" + data[0]["port"]+ "/" + data[0]["database"]
client   = MongoClient(uri)
mydatabse = client[data[0]["database"]]
mycollection = mydatabse["product_categories"]
mongo_data = pd.DataFrame(list(mycollection.find()))
mongo_data.head()

CLeaning the data

In [0]:
from pyspark.sql.functions import col,to_date,datediff, current_date,when

In [0]:
import pandas as pd
def clean_dataframe(df, name):
    ## dropping duplicate records
    print(f"Cleaning {name} of records {df.count()} ")
    return df.dropDuplicates().na.drop('all')


#order = dfs["order_df"]
orders_df= clean_dataframe(dfs["order_df"], "order")
print (orders_df.count())




## converting datetime columns into date only

In [0]:
orders_df = orders_df.withColumn('order_purchase_timestamp', to_date(col('order_purchase_timestamp'), 'mm/dd/yyy'))\
                      .withColumn('order_delivered_customer_date', to_date(col('order_delivered_customer_date'), 'mm/dd/yyyy'))\
                      .withColumn('order_estimated_delivery_date', to_date(col('order_estimated_delivery_date'), 'mm/dd/yyyy'))


In [0]:
## calculating delivery days and delay times
orders_df = orders_df.withColumn('actual_delivery_time', datediff('order_delivered_customer_date', 'order_purchase_timestamp'))
orders_df = orders_df.withColumn('estimated_delivery_time', datediff('order_estimated_delivery_date', 'order_purchase_timestamp'))
orders_df = orders_df.withColumn('delay time', col('actual_delivery_time')- col('estimated_delivery_time'))


In [0]:
display(orders_df.tail(5))

In [0]:
order_customer_df  = orders_df.join(dfs["customer_df"], on = 'customer_id', how = "left")


In [0]:
order_payment_df = order_customer_df.join(dfs["payment_df"], on = 'order_id', how = 'left')


In [0]:
orders_item_df = order_payment_df.join(dfs["order_item_df"], on = "order_id", how = "left")


In [0]:
order_items_product_df = orders_item_df.join(dfs["product_df"], on = "product_id", how = "left")

In [0]:
final_df = order_items_product_df.join(dfs["seller_df"], on="seller_id", how = "left" )
display(final_df.head(5))

drop data field ID from Mogodata dataframe

In [0]:
mongo_data.drop("_id",axis=1,inplace=True)
product_category_df = spark.createDataFrame(mongo_data)

In [0]:
final_df = final_df.join(product_category_df,on="product_category_name", how = "left")
display(final_df.head(5))

In [0]:
def remove_duplicate_columns(df):
    columns = df.columns

    seen_columns = set()
    columns_to_drop = []

    for column in columns:
        if column in seen_columns:
            columns_to_drop.append(column)
        else:
            seen_columns.add(column)
    
    df_cleaned = df.drop(*columns_to_drop)
    return df_cleaned

final_df = remove_duplicate_columns(final_df)

In [0]:
final_df.write.mode("overwrite").parquet("/Volumes/workspace/default/silver")